# WhoCry — Pelatihan di Google Colab

Notebook ini menjalankan eksperimen F4–F6 (baseline, LSTM/GRU, pelatihan ulang).

**Persiapan (sekali saja):**
1. Unggah `data/processed/hourly.parquet` dari laptop ke Google Drive di `MyDrive/whocry/hourly.parquet`.
2. Di Colab: *Runtime → Change runtime type → T4 GPU*.

Semua hasil disimpan di `MyDrive/whocry/outputs/`, jadi aman walaupun sesi Colab terputus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/whocry'
!test -d akuKehujanan || git clone -q https://github.com/SirojMun/akuKehujanan.git
%cd akuKehujanan
!git pull -q
!pip install -q -e .

In [ ]:
import torch, pandas as pd, numpy as np, sys
sys.path.insert(0, 'scripts')
from run_experiment import load, run
from whocry.train import TrainConfig

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA — ganti runtime ke GPU')
hourly, locations = load(f'{DRIVE}/hourly.parquet')
OUT = f'{DRIVE}/outputs'
print(f'{len(hourly):,} baris, {hourly.adm2.nunique()} lokasi, {hourly.time.min()} – {hourly.time.max()}')

## 1. Baseline (F4)
Persistence dan seasonal naive tidak dilatih, jadi cepat. Regresi linier dilatih untuk setiap varian spasial.

In [ ]:
for name in ['persistence', 'seasonal_naive']:
    run(TrainConfig(model=name), 'S0', hourly, locations, OUT)
for variant in ['S0', 'S1', 'S2', 'S3']:
    run(TrainConfig(model='linear', lr=1e-3, max_epochs=20, max_train_samples=1_000_000),
        variant, hourly, locations, OUT)

## 2. Pencarian hiperparameter (F5)
*Random search* kecil pada GRU varian S1 dengan subsampel data latih agar cepat.
Konfigurasi terbaik (val_loss terendah) dipakai untuk semua model RNN.

In [ ]:
import itertools, random, json, glob
random.seed(0)
space = dict(hidden=[32, 64, 128], layers=[1, 2], dropout=[0.0, 0.1, 0.2], lr=[1e-3, 3e-3])
configs = random.sample(list(itertools.product(*space.values())), 12)
search = []
for hidden, layers, dropout, lr in configs:
    cfg = TrainConfig(model='gru', hidden=hidden, layers=layers, dropout=dropout, lr=lr,
                      max_epochs=15, max_train_samples=300_000)
    out, metrics, _ = run(cfg, 'S1', hourly, locations, f'{OUT}/search')
    hist = pd.read_csv(out / 'history.csv')
    search.append(dict(hidden=hidden, layers=layers, dropout=dropout, lr=lr, val_loss=hist.val_loss.min()))
search = pd.DataFrame(search).sort_values('val_loss')
search.to_csv(f'{OUT}/search/summary.csv', index=False)
search

In [ ]:
best = search.iloc[0]
BEST = dict(hidden=int(best.hidden), layers=int(best.layers), dropout=float(best.dropout), lr=float(best.lr))
print('Konfigurasi terbaik:', BEST)

## 3. Matriks eksperimen utama (F5)
LSTM dan GRU × S0–S3 × 3 seed. Cukup lama; bisa dijalankan bertahap karena setiap run tersimpan sendiri di Drive.

In [ ]:
done = {json.load(open(p))['run_id'].split('_', 1)[1] for p in glob.glob(f'{OUT}/main/*/config.json')}
for model in ['lstm', 'gru']:
    for variant in ['S0', 'S1', 'S2', 'S3']:
        for seed in [0, 1, 2]:
            if f'{model}_{variant}_s{seed}' in done:
                continue
            cfg = TrainConfig(model=model, seed=seed, max_epochs=30, max_train_samples=1_000_000, **BEST)
            run(cfg, variant, hourly, locations, f'{OUT}/main')

## 4. Rekap hasil

In [ ]:
rows = []
for cfg_path in glob.glob(f'{OUT}/*/*/config.json') + glob.glob(f'{OUT}/*/config.json'):
    c = json.load(open(cfg_path))
    m = pd.read_csv(cfg_path.replace('config.json', 'metrics.csv'))
    m = m[m.horizon.isin([1, 3, 6, 12, 24])]
    for _, r in m.iterrows():
        rows.append(dict(model=c['model'], variant=c['variant'], seed=c['seed'], **r))
res = pd.DataFrame(rows)
res = res[~res.model.isna()]
table = (res.groupby(['target', 'model', 'variant', 'horizon']).mae.agg(['mean', 'std'])
            .round(3).unstack('horizon'))
table.to_csv(f'{OUT}/rekap_mae.csv')
table

## 5. Simulasi pelatihan ulang (F6)
Model terbaik dari langkah 3 (isi `BEST_MODEL` dan `BEST_VARIANT` sesuai rekap), lalu R0–R3 disimulasikan minggu per minggu pada periode uji.

In [ ]:
from whocry.features.build import prepare
from whocry.retrain import RetrainConfig, simulate
from whocry.train import make_model, fit

BEST_MODEL, BEST_VARIANT = 'gru', 'S2'   # ← sesuaikan dengan rekap
cfg = TrainConfig(model=BEST_MODEL, seed=0, max_epochs=30, max_train_samples=1_000_000, **BEST)
data, index, meta = prepare(hourly, locations, BEST_VARIANT)
base, _ = fit(cfg, data, index)
results = {}
for strategy in ['R0', 'R1', 'R2', 'R3']:
    weeks = simulate(base, lambda: make_model(cfg, data), data, meta, RetrainConfig(strategy=strategy))
    weeks.to_csv(f'{OUT}/retrain_{strategy}.csv', index=False)
    results[strategy] = weeks

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
for s, w in results.items():
    ax.plot(w.start, w.norm_mae.rolling(4, min_periods=1).mean(), label=s)
ax.set_ylabel('MAE ternormalisasi (rata-rata 4 minggu)'); ax.legend(); ax.grid(alpha=.3)
fig.savefig(f'{OUT}/retrain_mae.png', dpi=150, bbox_inches='tight')
pd.DataFrame({s: dict(mae=w.norm_mae.mean(), n_update=int(w.updated.sum()),
                      detik_update=w.update_seconds.sum()) for s, w in results.items()}).T